# 10_review_response
Additional analyses addressing referee concerns (IME major revision):
- C3  differential-detection threat: utilisation-anchored disease definition
      (self-report AND chronic-care outpatient visits) for the HTN causal axis.
- C4  exposure/outcome window overlap: lagged design (exposure t->t+1, onset
      t+1->t+2 among those disease-free at t+1) as the primary HTN specification.
- C5  richer propensity model (behaviours, baseline utilisation, education).
- C8  two-year realisability distribution alongside the one-year one.
- C12 Table 1: cohort characteristics by treatment status.
- C13 test for a time trend in the equity (attainment) gap.
- C7  drinking-harmonisation sensitivity (exclude 2023-2024 waves).
- C1  actuarial payload: translate the attainment gap into a stylised
      selection/loss distortion.
All figures greyscale, dpi 600, png+pdf; tables to results/tables.

In [1]:
%run 00_config.ipynb

PROJ_DIR: /home/claude/recourse_khp
1y pairs: [(2019, 2020), (2020, 2021), (2021, 2022), (2022, 2023), (2023, 2024)]
2y pairs: [(2019, 2021), (2020, 2022), (2021, 2023), (2022, 2024)]
registry loaded
helpers loaded
00_config ready


In [2]:
import statsmodels.api as sm, statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.stats import mannwhitneyu, linregress
rng=np.random.default_rng(42)
panel=pd.read_parquet(os.path.join(DATA_DIR,"panel_long.parquet"))
tr =pd.read_parquet(os.path.join(DATA_DIR,"transitions_1y.parquet"))
tr2=pd.read_parquet(os.path.join(DATA_DIR,"transitions_2y.parquet"))
for df in (tr,tr2): df["d_BMI"]=df["BMI_t1"]-df["BMI_t0"]
print("loaded")

loaded


In [3]:
def ipw(d,cols):
    Xp=d[cols].copy()
    for c in cols: Xp[c]=pd.to_numeric(Xp[c],errors="coerce"); Xp[c]=Xp[c].fillna(Xp[c].median())
    e=Pipeline([("sc",StandardScaler()),("lr",LogisticRegression(max_iter=1000))]).fit(Xp,d["TREAT"]).predict_proba(Xp)[:,1]
    e=np.clip(e,0.02,0.98); pt=d["TREAT"].mean()
    return np.where(d["TREAT"]==1,pt/e,(1-pt)/(1-e))
def fit2(d,sw,extra=""):
    f="onset ~ TREAT + bmi0 + age0 + male + C(year0)"+extra
    m=smf.glm(f,data=d,family=sm.families.Binomial(),freq_weights=sw).fit(cov_type="HC1")
    return m, np.exp(m.params["TREAT"]), np.exp(m.conf_int().loc["TREAT"]).values, m.pvalues["TREAT"]
print("estimators ready")

estimators ready


In [4]:
# Merge baseline (t0) behaviour + utilisation for richer PS (C5) and definitions (C3)
base=panel[[KEY,"year","OUGUN","INGUN","PA_REG_placeholder" if False else "PA_REG","ALC_FREQ","EDU"]].copy() \
     if "PA_REG" in panel.columns else None
addcols=[c for c in ["OUGUN","INGUN","PA_REG","ALC_FREQ","EDU"] if c in panel.columns]
b0=panel[[KEY,"year"]+addcols].rename(columns={"year":"year0",**{c:c+"_t0b" for c in addcols}})
trR=tr.merge(b0,on=[KEY,"year0"],how="left")
print("rich baseline merged:", [c for c in trR.columns if c.endswith("_t0b")])

rich baseline merged: ['OUGUN_t0b', 'INGUN_t0b', 'PA_REG_t0b', 'ALC_FREQ_t0b', 'EDU_t0b']


In [5]:
# Build HTN trial (primary, contemporaneous) with rich PS
def build(df,target,tau=None):
    d=df[df[f"{target}_atrisk"]==1].dropna(subset=["BMI_t0","BMI_t1",f"{target}_onset","age_t0","SEX_t0","H_INC_TOT_t0"]).copy()
    red=-(d["BMI_t1"]-d["BMI_t0"])
    if tau is None: tau=float(red[red>0].quantile(0.75))
    d["TREAT"]=(red>=tau).astype(int); d["male"]=(d["SEX_t0"]=="M").astype(int)
    d["onset"]=d[f"{target}_onset"].astype(int); d["bmi0"]=d["BMI_t0"]; d["age0"]=d["age_t0"]
    d["inc0"]=d["H_INC_TOT_t0"].fillna(d["H_INC_TOT_t0"].median()); d["year0"]=d["year0"].astype("category")
    return d,tau
H,tauH=build(trR,"HTN")
base_ps=["bmi0","age0","male","inc0"]
rich_ps=base_ps+[c for c in ["PA_REG_t0b","ALC_FREQ_t0b","OUGUN_t0b","EDU_t0b"] if c in H.columns]
sw_base=ipw(H,base_ps); sw_rich=ipw(H,rich_ps)
_,orb,cib,pb=fit2(H,sw_base); _,orr,cir,pr=fit2(H,sw_rich)
c5=pd.DataFrame([
 {"PS_model":"baseline (4 covs)","OR":round(orb,3),"lo":round(cib[0],3),"hi":round(cib[1],3),"p":round(pb,4)},
 {"PS_model":"rich (+behaviour,util,edu)","OR":round(orr,3),"lo":round(cir[0],3),"hi":round(cir[1],3),"p":round(pr,4)},
])
savetable(c5,"t10_c5_rich_ps", index=False); print(c5.to_string(index=False))

saved: t10_c5_rich_ps.csv
                  PS_model    OR    lo    hi      p
         baseline (4 covs) 0.608 0.450 0.822 0.0012
rich (+behaviour,util,edu) 0.610 0.452 0.824 0.0013


In [6]:
# C3: differential-detection robustness.
# Stronger onset definition = self-reported onset AND evidence of chronic-care use.
# We approximate chronic-care engagement at t+1 by above-median outpatient visits,
# so a "detected+managed" onset is less sensitive to pure self-report differences.
if "OUGUN_t1" in tr.columns:
    trA=tr.copy()
    trA["ou1"]=pd.to_numeric(trA["OUGUN_t1"],errors="coerce")
    med_ou=trA.loc[trA["ou1"]>0,"ou1"].median()
    HA,tauA=build(trA,"HTN")
    HA["ou1"]=pd.to_numeric(HA["OUGUN_t1"],errors="coerce") if "OUGUN_t1" in HA.columns else np.nan
    # anchored onset: onset==1 AND outpatient use above cohort median (proxy for managed dx)
    HA["onset_anchored"]=((HA["onset"]==1)&(HA["ou1"]>=med_ou)).astype(int)
    swA=ipw(HA,base_ps)
    # refit with anchored outcome
    m=smf.glm("onset_anchored ~ TREAT + bmi0 + age0 + male + C(year0)",data=HA,
              family=sm.families.Binomial(),freq_weights=swA).fit(cov_type="HC1")
    orA_=np.exp(m.params["TREAT"]); ciA_=np.exp(m.conf_int().loc["TREAT"]).values
    c3=pd.DataFrame([
      {"outcome_def":"self-report onset (primary)","OR":round(orb,3),"lo":round(cib[0],3),"hi":round(cib[1],3)},
      {"outcome_def":"utilisation-anchored onset","OR":round(orA_,3),"lo":round(ciA_[0],3),"hi":round(ciA_[1],3)},
    ])
    savetable(c3,"t10_c3_anchored_outcome", index=False); print(c3.to_string(index=False))

saved: t10_c3_anchored_outcome.csv
                outcome_def    OR    lo    hi
self-report onset (primary) 0.608 0.450 0.822
 utilisation-anchored onset 0.654 0.461 0.928


In [7]:
# C4: lagged design to break exposure/outcome window overlap.
# Exposure realised over (t0->t1); outcome = onset over (t1->t2) among those
# disease-free at t1. Implement by chaining wave pairs on PIDWON.
seq=list(WAVES.keys())
rows=[]
for i in range(len(seq)-2):
    w0,w1,w2=seq[i],seq[i+1],seq[i+2]
    a=panel[panel.wave==w0].set_index(KEY)
    b=panel[panel.wave==w1].set_index(KEY)
    c=panel[panel.wave==w2].set_index(KEY)
    idx=a.index.intersection(b.index).intersection(c.index)
    df=pd.DataFrame(index=idx)
    df["bmi0"]=a.loc[idx,"BMI"]; df["bmi1"]=b.loc[idx,"BMI"]
    df["htn1"]=b.loc[idx,"HTN_dx"]; df["htn2"]=c.loc[idx,"HTN_dx"]
    df["age0"]=a.loc[idx,"age"]; df["male"]=(a.loc[idx,"SEX"]=="M").astype(int)
    df["inc0"]=a.loc[idx,"H_INC_TOT"]; df["year0"]=WAVES[w0]
    rows.append(df.reset_index())
L=pd.concat(rows,ignore_index=True)
L=L.dropna(subset=["bmi0","bmi1","htn1","htn2","age0"])
L=L[L["htn1"]==0]                       # disease-free at t1 (exposure already realised)
red=-(L["bmi1"]-L["bmi0"]); tau=float(red[red>0].quantile(0.75))
L["TREAT"]=(red>=tau).astype(int); L["onset"]=(L["htn2"]==1).astype(int)
L["inc0"]=L["inc0"].fillna(L["inc0"].median()); L["year0"]=L["year0"].astype("category")
swL=ipw(L,["bmi0","age0","male","inc0"]); _,orL,ciL,pL=fit2(L,swL)
c4=pd.DataFrame([{"design":"lagged (exp t0->t1, onset t1->t2)","n":len(L),
                 "treated":int(L.TREAT.sum()),"onset":int(L.onset.sum()),
                 "OR":round(orL,3),"lo":round(ciL[0],3),"hi":round(ciL[1],3),"p":round(pL,4)}])
savetable(c4,"t10_c4_lagged", index=False); print(c4.to_string(index=False))

saved: t10_c4_lagged.csv
                           design     n  treated  onset   OR    lo    hi      p
lagged (exp t0->t1, onset t1->t2) 21939     1733    695 1.03 0.779 1.361 0.8379


In [8]:
# C8: realisability at 2-year horizon vs 1-year
def realised_reduction(df):
    return (-(df["BMI_t1"]-df["BMI_t0"])).dropna()
r1=realised_reduction(tr[tr["HTN_atrisk"]==1]); r2=realised_reduction(tr2[tr2["HTN_atrisk"]==1])
presc_med=4.34   # from t03 (median prescribed BMI reduction)
c8=pd.DataFrame([
 {"horizon":"1y","q50":round(r1.quantile(.5),2),"q90":round(r1.quantile(.9),2),
  "achieve_median_presc_%":round(100*(r1>=presc_med).mean(),2)},
 {"horizon":"2y","q50":round(r2.quantile(.5),2),"q90":round(r2.quantile(.9),2),
  "achieve_median_presc_%":round(100*(r2>=presc_med).mean(),2)},
])
savetable(c8,"t10_c8_realisability_2y", index=False); print(c8.to_string(index=False))

saved: t10_c8_realisability_2y.csv
horizon  q50  q90  achieve_median_presc_%
     1y -0.0 1.06                    0.49
     2y -0.0 1.32                    0.68


In [9]:
# C12: cohort characteristics by treatment status (HTN primary trial)
def tab1(d):
    rows=[]
    def add(name,tr1,tr0,fmt="{:.2f}"):
        rows.append({"variable":name,"treated":fmt.format(tr1),"control":fmt.format(tr0)})
    g1=d[d.TREAT==1]; g0=d[d.TREAT==0]
    add("n",len(g1),len(g0),"{:.0f}")
    add("BMI (t0)",g1.bmi0.mean(),g0.bmi0.mean())
    add("Age (t0)",g1.age0.mean(),g0.age0.mean())
    add("Male %",100*g1.male.mean(),100*g0.male.mean())
    add("Household income (t0)",g1.inc0.mean(),g0.inc0.mean(),"{:.0f}")
    add("HTN onset %",100*g1.onset.mean(),100*g0.onset.mean())
    return pd.DataFrame(rows)
t1=tab1(H); savetable(t1,"t10_c12_table1", index=False); print(t1.to_string(index=False))

saved: t10_c12_table1.csv
             variable treated control
                    n    2343   27361
             BMI (t0)   25.33   23.15
             Age (t0)   51.68   53.37
               Male %   42.85   43.80
Household income (t0)    5389    5635
          HTN onset %    2.52    3.03


In [10]:
# C13: is the young-vs-old attainment gap trending over wave pairs?
atr=tr[tr["HTN_atrisk"]==1].copy(); atr["red"]=-(atr["BMI_t1"]-atr["BMI_t0"])
tauA=float(atr["red"][atr["red"]>0].quantile(0.75)); atr["attain"]=(atr["red"]>=tauA).astype(int)
gaps=[]
for pr,g in atr.dropna(subset=["AGEG_t0"]).groupby("pair",observed=True):
    y=g.loc[g["AGEG_t0"]=="19-29","attain"].mean(); o=g.loc[g["AGEG_t0"]=="70+","attain"].mean()
    if pd.notna(y) and pd.notna(o): gaps.append({"pair":pr,"gap":y-o,"t":len(gaps)})
G=pd.DataFrame(gaps)
lr=linregress(G["t"],G["gap"])
c13=pd.DataFrame([{"slope_per_wave":round(lr.slope,4),"p_value":round(lr.pvalue,3),
                   "r_squared":round(lr.rvalue**2,3),"mean_gap":round(G["gap"].mean(),3)}])
savetable(c13,"t10_c13_gap_trend", index=False)
print(G.to_string(index=False)); print(c13.to_string(index=False))

saved: t10_c13_gap_trend.csv
     pair       gap  t
2019_2020 -0.013208  0
2020_2021  0.020049  1
2021_2022  0.022777  2
2022_2023  0.019928  3
2023_2024 -0.008339  4
 slope_per_wave  p_value  r_squared  mean_gap
          0.001    0.889      0.008     0.008


In [11]:
# C7: exclude 2023-2024 (D1_2 rename) -> rerun HTN axis-2 without those pairs
trD=trR[~trR["pair"].isin(["2022_2023","2023_2024"])].copy()
HD,_=build(trD,"HTN")
swD=ipw(HD,base_ps); _,orD,ciD,pD=fit2(HD,swD)
c7=pd.DataFrame([
 {"sample":"all waves","OR":round(orb,3),"lo":round(cib[0],3),"hi":round(cib[1],3)},
 {"sample":"excl. 2023-2024 (drinking rename)","OR":round(orD,3),"lo":round(ciD[0],3),"hi":round(ciD[1],3)},
])
savetable(c7,"t10_c7_drinking_sensitivity", index=False); print(c7.to_string(index=False))

saved: t10_c7_drinking_sensitivity.csv
                           sample    OR    lo    hi
                        all waves 0.608 0.450 0.822
excl. 2023-2024 (drinking rename) 0.732 0.513 1.044


In [12]:
# C1: stylised actuarial translation of the attainment gap.
# If recourse is offered as a premium-relief pathway (approved if realised), then
# groups with lower attainment obtain relief less often. We quantify the resulting
# expected-premium gap under a stylised two-type model.
# attain rates by income quintile:
aq=(atr.dropna(subset=["INCQ_t0"]).groupby("INCQ_t0",observed=True)["attain"].mean()
      .reindex(["Q1","Q2","Q3","Q4","Q5"]))
# stylised: declined applicants face surcharge s on base premium P; realising recourse
# removes surcharge. Expected surcharge borne by group g = s*(1-attain_g).
s=0.20   # 20% surcharge (illustrative)
exp_surch=s*(1-aq)
gap=(exp_surch.max()-exp_surch.min())
c1=pd.DataFrame({"income_quintile":aq.index,"attain_rate":aq.round(3).values,
                 "expected_surcharge_frac":exp_surch.round(3).values})
savetable(c1,"t10_c1_actuarial_gap", index=False)
print(c1.to_string(index=False))
print(f"\nStylised expected-premium-relief gap (Q1 vs Q5) = {gap*100:.1f}% of base premium at s={s:.0%}")

saved: t10_c1_actuarial_gap.csv
income_quintile  attain_rate  expected_surcharge_frac
             Q1        0.086                    0.183
             Q2        0.076                    0.185
             Q3        0.072                    0.186
             Q4        0.077                    0.185
             Q5        0.071                    0.186

Stylised expected-premium-relief gap (Q1 vs Q5) = 0.3% of base premium at s=20%


In [13]:
# C1 figure: expected surcharge by income quintile (attainment-driven)
fig,ax=plt.subplots(figsize=(5.4,4.0))
ax.bar(aq.index.astype(str),(s*(1-aq)*100).values,color="#777777",edgecolor="black",linewidth=0.5)
ax.set_xlabel("Household income quintile")
ax.set_ylabel("Expected surcharge borne (% of base premium)")
savefig(fig,"f10_c1_actuarial_gap"); plt.close(fig)
print("actuarial figure saved")

saved: f10_c1_actuarial_gap.png / f10_c1_actuarial_gap.pdf
actuarial figure saved
